In [1]:
import pandas as pd

cell_data = pd.read_csv("app_data/cell_data.csv", index_col=0)
regulon_activity = pd.read_csv("app_data/regulon_activity.csv", index_col=0)
regulon_summary = pd.read_csv("app_data/regulon_summary.csv", index_col=0)
program_loadings = pd.read_csv("app_data/program_loadings.csv", index_col=0)
program_regulons = pd.read_csv("app_data/program_regulons.csv")
program_enrichment = pd.read_csv("app_data/program_enrichment.csv")

print("cell_data:", cell_data.shape)
print("regulon_activity:", regulon_activity.shape)
print("regulon_summary:", regulon_summary.shape)
print("program_loadings:", program_loadings.shape)
print("program_regulons:", program_regulons.shape)
print("program_enrichment:", program_enrichment.shape)

cell_data: (24073, 12)
regulon_activity: (24073, 225)
regulon_summary: (225, 8)
program_loadings: (225, 8)
program_regulons: (80, 4)
program_enrichment: (150, 15)


In [2]:
import sys
print(sys.executable)

!{sys.executable} -m pip install -U nbformat

/Users/serafimaabakumova/miniforge3/envs/scrna/bin/python


In [3]:
import plotly.express as px

selected_regulon = regulon_activity.columns[0]

plot_data = cell_data[
    ["reg_umap1", "reg_umap2"]
].copy()

plot_data["activity"] = regulon_activity[selected_regulon]

fig = px.scatter(
    plot_data,
    x="reg_umap1",
    y="reg_umap2",
    color="activity",
    title=f"Regulatory UMAP — {selected_regulon}",
    labels={
        "reg_umap1": "Regulatory UMAP 1",
        "reg_umap2": "Regulatory UMAP 2",
        "activity": "AUCell activity"
    },
    opacity=0.65,
    render_mode="webgl"
)

fig.update_traces(
    marker={"size": 3}
)

fig.update_layout(
    width=900,
    height=650
)

fig.show()

In [5]:
import ipywidgets as widgets
import plotly.express as px

# Dropdown containing all 225 regulons
regulon_selector = widgets.Dropdown(
    options=regulon_activity.columns.tolist(),
    description="Regulon:",
    style={"description_width": "initial"}
)


def plot_regulon(selected_regulon):

    # Regulatory UMAP coordinates for every cell
    plot_data = cell_data[
        ["reg_umap1", "reg_umap2"]
    ].copy()

    # AUCell activity of the selected regulon
    plot_data["activity"] = regulon_activity.loc[
        plot_data.index,
        selected_regulon
    ]

    # Interactive regulatory UMAP
    fig = px.scatter(
        plot_data,
        x="reg_umap1",
        y="reg_umap2",
        color="activity",

        # Same interpretable color scale for every regulon
        color_continuous_scale="Viridis",

        title=f"Regulatory UMAP — {selected_regulon}",

        labels={
            "reg_umap1": "Regulatory UMAP 1",
            "reg_umap2": "Regulatory UMAP 2",
            "activity": "AUCell activity"
        },

        opacity=0.65,
        render_mode="webgl"
    )

    # Small points because we have 24,073 cells
    fig.update_traces(
        marker={"size": 3}
    )

    # Figure appearance + explicit activity scale
    fig.update_layout(
        width=900,
        height=650,
        coloraxis_colorbar=dict(
            title="AUCell<br>activity",
            thickness=18,
            len=0.75
        )
    )

    fig.show()


# Connect the dropdown to the plotting function
widgets.interact(
    plot_regulon,
    selected_regulon=regulon_selector
)

interactive(children=(Dropdown(description='Regulon:', options=('AR(+)', 'ARID3A(+)', 'ATF3(+)', 'BACH2(+)', '…

<function __main__.plot_regulon(selected_regulon)>

In [6]:
# Toggle between gene-expression and regulatory representations
view_selector = widgets.ToggleButtons(
    options=["Expression", "Regulatory"],
    description="View:",
    style={"description_width": "initial"}
)


def plot_landscape(view):

    # Choose which precomputed UMAP coordinates to display
    if view == "Expression":
        x = "expr_umap1"
        y = "expr_umap2"
        title = "Gene Expression Landscape"
        x_label = "Expression UMAP 1"
        y_label = "Expression UMAP 2"

    else:
        x = "reg_umap1"
        y = "reg_umap2"
        title = "Regulatory Landscape"
        x_label = "Regulatory UMAP 1"
        y_label = "Regulatory UMAP 2"

    # Plot the same cells using the selected representation
    fig = px.scatter(
        cell_data,
        x=x,
        y=y,
        title=title,
        labels={
            x: x_label,
            y: y_label
        },
        opacity=0.55,
        render_mode="webgl"
    )

    fig.update_traces(
        marker={"size": 3}
    )

    fig.update_layout(
        width=900,
        height=650
    )

    fig.show()


# Connect buttons to plotting function
widgets.interact(
    plot_landscape,
    view=view_selector
)

interactive(children=(ToggleButtons(description='View:', options=('Expression', 'Regulatory'), style=ToggleBut…

<function __main__.plot_landscape(view)>

In [7]:
print(cell_data.columns.tolist())

['expr_umap1', 'expr_umap2', 'reg_umap1', 'reg_umap2', 'Program_1', 'Program_2', 'Program_3', 'Program_4', 'Program_5', 'Program_6', 'Program_7', 'Program_8']


In [8]:
# List of the 8 NMF regulatory components
program_columns = [
    "Program_1",
    "Program_2",
    "Program_3",
    "Program_4",
    "Program_5",
    "Program_6",
    "Program_7",
    "Program_8"
]

# Dropdown for selecting a component
program_selector = widgets.Dropdown(
    options=program_columns,
    description="Component:",
    style={"description_width": "initial"}
)


def plot_program(selected_program):

    # Regulatory UMAP coordinates
    plot_data = cell_data[
        ["reg_umap1", "reg_umap2", selected_program]
    ].copy()

    # Interactive UMAP colored by NMF component activity
    fig = px.scatter(
        plot_data,
        x="reg_umap1",
        y="reg_umap2",
        color=selected_program,
        color_continuous_scale="Viridis",

        title=f"Regulatory UMAP — {selected_program}",

        labels={
            "reg_umap1": "Regulatory UMAP 1",
            "reg_umap2": "Regulatory UMAP 2",
            selected_program: "NMF activity"
        },

        opacity=0.65,
        render_mode="webgl"
    )

    fig.update_traces(
        marker={"size": 3}
    )

    fig.update_layout(
        width=900,
        height=650,
        coloraxis_colorbar=dict(
            title="NMF<br>activity",
            thickness=18,
            len=0.75
        )
    )

    fig.show()


# Connect dropdown to plotting function
widgets.interact(
    plot_program,
    selected_program=program_selector
)

interactive(children=(Dropdown(description='Component:', options=('Program_1', 'Program_2', 'Program_3', 'Prog…

<function __main__.plot_program(selected_program)>

In [ ]:
def show_component_details(selected_program):

    # Get regulon loadings for the selected NMF component
    loadings = program_loadings[selected_program].sort_values(
        ascending=False
    )

    # Select the 10 strongest contributing regulons
    top_regulons = loadings.head(10).reset_index()

    top_regulons.columns = [
        "Regulon",
        "Loading"
    ]

    print(f"What defines {selected_program}?")
    print()
    print("Top contributing regulons:")

    display(top_regulons)


widgets.interact(
    show_component_details,
    selected_program=program_selector
)

interactive(children=(Dropdown(description='Component:', index=7, options=('Program_1', 'Program_2', 'Program_…

<function __main__.show_component_details(selected_program)>

In [ ]:
def show_component_enrichment(selected_program):

    # Select enrichment results for this component
    result = program_enrichment[
        program_enrichment["program"] == selected_program
    ].copy()

    # Remove generic transcription-related terms
    generic_terms = [
        "transcription",
        "RNA metabolic",
        "RNA biosynthetic",
        "gene expression",
        "nucleobase-containing",
        "biosynthetic process",
        "metabolic process"
    ]

    specific = result[
        ~result["name"].str.contains(
            "|".join(generic_terms),
            case=False,
            regex=True,
            na=False
        )
    ].copy()

    print(f"Functional enrichment — {selected_program}")
    print()

    if specific.empty:
        print("No specific enriched pathways remained after filtering generic terms.")

    else:
        display(
            specific[
                ["source", "name", "p_value", "intersection_size"]
            ].head(10)
        )


widgets.interact(
    show_component_enrichment,
    selected_program=program_selector
)

interactive(children=(Dropdown(description='Component:', index=7, options=('Program_1', 'Program_2', 'Program_…

<function __main__.show_component_enrichment(selected_program)>

In [ ]:
component_selector = widgets.Dropdown(
    options=[f"Program_{i}" for i in range(1, 9)],
    description="Component:",
    style={"description_width": "initial"}
)


def component_explorer(selected_program):

    # --------------------------------------------------
    # 1. REGULATORY UMAP
    # --------------------------------------------------

    plot_data = cell_data[
        ["reg_umap1", "reg_umap2", selected_program]
    ].copy()

    fig = px.scatter(
        plot_data,
        x="reg_umap1",
        y="reg_umap2",
        color=selected_program,
        color_continuous_scale="Viridis",
        title=f"Regulatory UMAP — {selected_program}",
        labels={
            "reg_umap1": "Regulatory UMAP 1",
            "reg_umap2": "Regulatory UMAP 2",
            selected_program: "NMF activity"
        },
        opacity=0.65,
        render_mode="webgl"
    )

    fig.update_traces(marker={"size": 3})

    fig.update_layout(
        width=900,
        height=650,
        coloraxis_colorbar=dict(
            title="NMF<br>activity",
            thickness=18,
            len=0.75
        )
    )

    fig.show()


    # --------------------------------------------------
    # 2. TOP CONTRIBUTING REGULONS
    # --------------------------------------------------

    print("\nWhat defines this component?")
    print("Top contributing regulons:")

    loadings = (
        program_loadings[selected_program]
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    loadings.columns = ["Regulon", "Loading"]

    display(loadings)


    # --------------------------------------------------
    # 3. FUNCTIONAL ENRICHMENT
    # --------------------------------------------------

    print("\nFunctional enrichment:")

    enrichment = program_enrichment[
        program_enrichment["program"] == selected_program
    ].copy()

    generic_terms = [
        "transcription",
        "RNA metabolic",
        "RNA biosynthetic",
        "gene expression",
        "nucleobase-containing",
        "biosynthetic process",
        "metabolic process"
    ]

    specific_enrichment = enrichment[
        ~enrichment["name"].str.contains(
            "|".join(generic_terms),
            case=False,
            regex=True,
            na=False
        )
    ].copy()

    if specific_enrichment.empty:

        print(
            "No specific functional enrichment identified.\n"
            "Significant enrichment was dominated by broad "
            "transcription/regulation terms, so no pathway-level "
            "interpretation was assigned."
        )

    else:

        display(
            specific_enrichment[
                [
                    "source",
                    "name",
                    "p_value",
                    "intersection_size"
                ]
            ].head(10)
        )


widgets.interact(
    component_explorer,
    selected_program=component_selector
)

interactive(children=(Dropdown(description='Component:', options=('Program_1', 'Program_2', 'Program_3', 'Prog…

<function __main__.component_explorer(selected_program)>

In [12]:
print(regulon_summary.columns.tolist())
print(regulon_summary.head())

['mean', 'median', 'std', 'q25', 'q75', 'q95', 'max', 'pct_nonzero']
               mean    median       std       q25       q75       q95  \
AR(+)      0.111238  0.109496  0.029899  0.102067  0.114018  0.117248   
ARID3A(+)  0.030979  0.029648  0.008198  0.026758  0.031484  0.046886   
ATF3(+)    0.013663  0.011545  0.007514  0.010638  0.012123  0.031214   
BACH2(+)   0.002341  0.000000  0.013542  0.000000  0.000000  0.000000   
BCL11A(+)  0.003069  0.000000  0.013740  0.000000  0.000000  0.040879   

                max  pct_nonzero  
AR(+)      0.744186    99.945998  
ARID3A(+)  0.287094    99.995846  
ATF3(+)    0.170790    99.937690  
BACH2(+)   0.226744     3.069829  
BCL11A(+)  0.301163     5.076226  


In [ ]:
regulon_selector_full = widgets.Dropdown(
    options=regulon_activity.columns.tolist(),
    description="Regulon:",
    style={"description_width": "initial"}
)


def regulon_explorer(selected_regulon):

    # --------------------------------------------------
    # 1. REGULATORY UMAP
    # --------------------------------------------------

    plot_data = cell_data[
        ["reg_umap1", "reg_umap2"]
    ].copy()

    plot_data["activity"] = regulon_activity.loc[
        plot_data.index,
        selected_regulon
    ]

    fig = px.scatter(
        plot_data,
        x="reg_umap1",
        y="reg_umap2",
        color="activity",
        color_continuous_scale="Viridis",
        title=f"Regulatory UMAP — {selected_regulon}",
        labels={
            "reg_umap1": "Regulatory UMAP 1",
            "reg_umap2": "Regulatory UMAP 2",
            "activity": "AUCell activity"
        },
        opacity=0.65,
        render_mode="webgl"
    )

    fig.update_traces(marker={"size": 3})

    fig.update_layout(
        width=900,
        height=650,
        coloraxis_colorbar=dict(
            title="AUCell<br>activity",
            thickness=18,
            len=0.75
        )
    )

    fig.show()


    # --------------------------------------------------
    # 2. REGULON STATISTICS
    # --------------------------------------------------

    stats = regulon_summary.loc[selected_regulon]

    prevalence = stats["pct_nonzero"]

    print("\nRegulon statistics")
    print(f"Prevalence:      {prevalence:.2f}% of cells")
    print(f"Mean activity:   {stats['mean']:.4f}")
    print(f"Median activity: {stats['median']:.4f}")
    print(f"Variability:     {stats['std']:.4f}")
    print(f"95th percentile: {stats['q95']:.4f}")
    print(f"Maximum:         {stats['max']:.4f}")


    # --------------------------------------------------
    # 3. ACTIVITY PATTERN
    # --------------------------------------------------

    print("\nActivity pattern")

    if prevalence <= 1:
        print(
            "Very rare / highly localized activity "
            f"({prevalence:.2f}% of cells)."
        )

    elif prevalence <= 5:
        print(
            "Rare activity "
            f"({prevalence:.2f}% of cells)."
        )

    elif prevalence <= 20:
        print(
            "Activity is present in a minority of cells "
            f"({prevalence:.2f}%)."
        )

    elif prevalence < 80:
        print(
            "Activity is distributed across a substantial "
            f"fraction of cells ({prevalence:.2f}%)."
        )

    else:
        print(
            "Broadly detected activity "
            f"({prevalence:.2f}% of cells)."
        )

    print(
        "\nNote: prevalence describes the fraction of cells with "
        "non-zero AUCell activity. It does not measure activity strength "
        "and does not by itself define a biological cell state."
    )


widgets.interact(
    regulon_explorer,
    selected_regulon=regulon_selector_full
)

interactive(children=(Dropdown(description='Regulon:', options=('AR(+)', 'ARID3A(+)', 'ATF3(+)', 'BACH2(+)', '…

<function __main__.regulon_explorer(selected_regulon)>

In [14]:
import requests
import pandas as pd

# Extract TF symbols from regulon names:
# FOXO1(+) -> FOXO1
tf_symbols = [
    regulon.replace("(+)", "")
    for regulon in regulon_activity.columns
]

url = "https://mygene.info/v3/query"

response = requests.post(
    url,
    data={
        "q": ",".join(tf_symbols),
        "scopes": "symbol",
        "species": "human",
        "fields": "symbol,name,summary,entrezgene"
    }
)

response.raise_for_status()

results = response.json()

annotation_rows = []

for item in results:

    query_symbol = item.get("query")

    annotation_rows.append({
        "TF": query_symbol,
        "symbol": item.get("symbol"),
        "full_name": item.get("name"),
        "function": item.get("summary"),
        "entrezgene": item.get("entrezgene"),
        "found": not item.get("notfound", False)
    })

tf_annotations = pd.DataFrame(annotation_rows)

# Keep one result per queried TF
tf_annotations = (
    tf_annotations
    .drop_duplicates(subset="TF")
    .set_index("TF")
)

display(tf_annotations.head())

print("TFs requested:", len(tf_symbols))
print("Annotations returned:", len(tf_annotations))
print("With function:", tf_annotations["function"].notna().sum())

,symbol,full_name,function,entrezgene,found
TF,,,,,
AR,AR,androgen receptor,The androgen receptor gene is more than 90 kb ...,367,True
ARID3A,ARID3A,AT-rich interaction domain 3A,This gene encodes a member of the ARID (AT-ric...,1820,True
ATF3,ATF3,activating transcription factor 3,This gene encodes a member of the mammalian ac...,467,True
BACH2,BACH2,BACH transcriptional regulator 2,Enables sequence-specific double-stranded DNA ...,60468,True
BCL11A,BCL11A,BCL11 transcription factor A,This gene encodes a C2H2 type zinc-finger prot...,53335,True


TFs requested: 225
Annotations returned: 225
With function: 225


In [15]:
tf_annotations.to_csv(
    "app_data/tf_annotations.csv"
)

print("Saved:", tf_annotations.shape)

Saved: (225, 5)
